# 🤖 Generative Artificial Intelligence — Lecture 1
## Foundations of Generative Modelling
### Likelihood · Posterior · Decision Theory · Generative Model

**Haydar Kılıç | Faculty of Engineering, Artificial Intelligence Engineering**

---
This notebook provides hands-on Python demonstrations of the theoretical concepts covered in the lecture slides.

**Requirements:** Python 3.9+, NumPy 1.24+, Matplotlib 3.5+, SciPy 1.9+, scikit-learn 1.0+

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy.stats import norm
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# NumPy version compatibility: trapezoid was called trapz before NumPy 2.0
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ Libraries loaded successfully!')
print(f'   NumPy   : {np.__version__}')
import matplotlib; print(f'   Matplotlib: {matplotlib.__version__}')
import scipy;      print(f'   SciPy   : {scipy.__version__}')
import sklearn;    print(f'   sklearn : {sklearn.__version__}')

---
## 📌 SECTION 1: Fundamental Concepts
### 1.1 Handwritten Digit Recognition — Training & Test Set Concepts

Each digit is represented as a **28×28 = 784**-pixel vector. 
Our goal: learn a function that maps the vector $\mathbf{x}$ to the digit class $(0, \ldots, 9)$.

We simulate this idea below:

In [ ]:
# Generate synthetic 28x28 images representing digits 0-9
np.random.seed(42)

def make_digit_image(digit, noise=0.15):
    """Simulate a simple digit image."""
    # FIX: nested np.random.seed() calls corrupted global state.
    # Isolated RNG via the modern NumPy Generator API.
    rng = np.random.default_rng(digit * 7)
    img = np.zeros((28, 28))
    img[4:24, 4:24] = rng.random((20, 20)) * 0.3
    cx, cy = 14, 14
    for i in range(28):
        for j in range(28):
            d = ((i - cx)**2 + (j - cy)**2) ** 0.5
            if abs(d - (5 + digit % 4)) < 2:
                img[i, j] = 0.9
    img += rng.standard_normal((28, 28)) * noise
    img = np.clip(img, 0, 1)
    return img

# Visualise all 10 digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Handwritten Digits from Training Set (Simulated)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = make_digit_image(i)
    ax.imshow(img, cmap='Blues')
    ax.set_title(f'Digit: {i}', fontsize=11)
    ax.axis('off')
    t = np.zeros(10, dtype=int)
    t[i] = 1
    # FIX: display np.array in a more readable format
    ax.set_xlabel(f't = {list(t)}', fontsize=6)

plt.tight_layout()
plt.show()

print(f'\n📐 Vector size: 28 × 28 = {28*28} pixels')
print('🎯 Goal: assign each image to one of the 10 digit classes (0-9)')

In [ ]:
# Conceptual illustration of train / validation / test split
np.random.seed(0)
N_total = 100
indices = np.arange(N_total)
np.random.shuffle(indices)

train_idx = indices[:70]
val_idx   = indices[70:85]
test_idx  = indices[85:]

fig, ax = plt.subplots(figsize=(12, 2))
ax.barh(0, 70, color='#2196F3', label='Training Set (70%)')
ax.barh(0, 15, left=70, color='#FF9800', label='Validation Set (15%)')
ax.barh(0, 15, left=85, color='#4CAF50', label='Test Set (15%)')
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('Data Fraction (%)')
ax.set_title('Dataset Splitting Strategy', fontweight='bold')
ax.legend(loc='lower right')

ax.text(35, 0, 'Learn weight coefficients w', ha='center', va='center', color='white', fontweight='bold')
ax.text(77.5, 0, 'Select M or\nlambda', ha='center', va='center', color='white', fontsize=9)
ax.text(92.5, 0, 'Measure\ngeneralisation', ha='center', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.show()

---
### 1.2 Curve Fitting and Polynomial Regression

Polynomial function:
$$y(x, \mathbf{w}) = \sum_{j=0}^{M} w_j x^j$$

Error function to minimise (Least Squares):
$$E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^{N} \{y(x_n, \mathbf{w}) - t_n\}^2$$

In [ ]:
# Generate data: sin(2πx) + noise
np.random.seed(42)
N = 10
x_train = np.linspace(0, 1, N)
t_train = np.sin(2 * np.pi * x_train) + np.random.normal(0, 0.3, N)

x_true = np.linspace(0, 1, 200)
t_true = np.sin(2 * np.pi * x_true)

def fit_polynomial(x, t, M):
    """Fit a degree-M polynomial and return the coefficients.
    
    Solution: w* = (X^T X)^{-1} X^T t  (Least Squares)
    """
    X = np.vander(x, M + 1, increasing=True)  # Vandermonde matrix
    w = np.linalg.lstsq(X, t, rcond=None)[0]
    return w

def poly_predict(x, w):
    """Polynomial prediction function."""
    M = len(w) - 1
    X = np.vander(x, M + 1, increasing=True)
    return X @ w

def rms_error(y_pred, t):
    """Root Mean Square Error: E_RMS = sqrt(mean((y-t)^2))"""
    return np.sqrt(np.mean((y_pred - t)**2))

# Visualisation for different polynomial degrees M
degrees = [0, 1, 3, 9]
colors  = ['#E91E63', '#9C27B0', '#2196F3', '#FF5722']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Polynomial Curve Fitting — Different Degrees M', fontsize=15, fontweight='bold')

for ax, M, color in zip(axes.flat, degrees, colors):
    w = fit_polynomial(x_train, t_train, M)
    y_pred_train = poly_predict(x_train, w)
    y_pred_true  = poly_predict(x_true, w)

    E_train = rms_error(y_pred_train, t_train)

    ax.plot(x_true, t_true, 'g-', linewidth=2, label='True: sin(2πx)', zorder=1)
    ax.scatter(x_train, t_train, s=80, facecolors='none', edgecolors='blue',
               linewidths=2, zorder=3, label='Training data')
    ax.plot(x_true, y_pred_true, color=color, linewidth=2.5,
            label=f'M={M} polynomial', zorder=2)

    # Draw residual lines
    for xn, tn, yn in zip(x_train, t_train, y_pred_train):
        ax.plot([xn, xn], [tn, yn], 'g--', alpha=0.4, linewidth=1)

    ax.set_title(f'M = {M}  |  $E_{{RMS}}$ = {E_train:.4f}', fontsize=13)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.legend(fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('t')

plt.tight_layout()
plt.show()

---
### 1.3 Overfitting and RMS Error Comparison

$$E_{RMS} = \sqrt{\frac{2E(\mathbf{w}^*)}{N}}$$

As M increases, training error decreases but test error rises → **Overfitting!**

In [ ]:
# Generate test data
np.random.seed(99)
N_test = 100
x_test = np.linspace(0, 1, N_test)
t_test = np.sin(2 * np.pi * x_test) + np.random.normal(0, 0.3, N_test)

# Compute training and test error for each degree M
max_degree = 9
train_errors = []
test_errors  = []

for M in range(max_degree + 1):
    w = fit_polynomial(x_train, t_train, M)

    y_train_pred = poly_predict(x_train, w)
    y_test_pred  = poly_predict(x_test, w)

    # Clip to prevent numerical overflow for high-degree polynomials
    test_pred_clipped = np.clip(y_test_pred, -10, 10)

    train_errors.append(rms_error(y_train_pred, t_train))
    test_errors.append(rms_error(test_pred_clipped, t_test))

# Plotting
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(max_degree + 1), train_errors, 'bo-', linewidth=2,
        markersize=8, label='Training Error')
ax.plot(range(max_degree + 1), test_errors,  'ro-', linewidth=2,
        markersize=8, label='Test Error')

ax.axvline(x=3, color='green', linestyle='--', alpha=0.7, label='Optimal M=3')
ax.set_xlabel('Polynomial Degree M', fontsize=13)
ax.set_ylabel('$E_{RMS}$', fontsize=13)
ax.set_title('Model Selection: Training and Test Error vs. Polynomial Degree', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_xticks(range(max_degree + 1))
ax.set_ylim(0, 1.2)

# Region annotations
ax.axvspan(0, 2, alpha=0.08, color='red')
ax.axvspan(3, 4, alpha=0.08, color='green')
ax.axvspan(5, 9, alpha=0.08, color='orange')

ax.text(1,   1.1, 'Underfitting', ha='center', color='red',        fontsize=10)
ax.text(3.5, 1.1, 'Ideal',                          ha='center', color='green',      fontsize=10)
ax.text(7,   1.1, 'Overfitting',                    ha='center', color='darkorange', fontsize=10)

plt.tight_layout()
plt.show()

print('\n📊 Error Table:')
print(f'{"M":>3} | {"Train E_RMS":>12} | {"Test E_RMS":>12}')
print('-' * 33)
for M in range(max_degree + 1):
    marker = ' <- Optimal' if M == 3 else ''
    print(f'{M:>3} | {train_errors[M]:>12.4f} | {test_errors[M]:>12.4f}{marker}')

---
### 1.4 Regularization

To prevent overfitting, we add a penalty term to the error function:

$$\tilde{E}(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^{N} \{y(x_n, \mathbf{w}) - t_n\}^2 + \frac{\lambda}{2} \|\mathbf{w}\|^2$$

As $\lambda$ grows, the model becomes simpler (smaller weights) but more biased.

In [ ]:
def fit_polynomial_regularized(x, t, M, lam):
    """Fit a polynomial using Ridge regression (L2 regularization).
    
    Solution: (X^T X + λI) w = X^T t
    """
    X = np.vander(x, M + 1, increasing=True)
    A = X.T @ X + lam * np.eye(M + 1)
    b = X.T @ t
    w = np.linalg.solve(A, b)
    return w

# M=9, various λ values
M = 9
lambdas = {
    r'$\ln\lambda = -18\ (\lambda \approx 0)$': np.exp(-18),
    r'$\ln\lambda = -5$':                        np.exp(-5),
    r'$\ln\lambda = 0\ (\lambda = 1)$':          1.0,
    r'$\ln\lambda = 3$':                         np.exp(3),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle(r'Regularization — M=9 Polynomial, Different $\lambda$ Values',
             fontsize=14, fontweight='bold')

for ax, (label, lam) in zip(axes.flat, lambdas.items()):
    w = fit_polynomial_regularized(x_train, t_train, M, lam)
    y_fit = poly_predict(x_true, w)
    y_fit = np.clip(y_fit, -2, 2)

    E_rms  = rms_error(poly_predict(x_train, w), t_train)
    w_norm = np.linalg.norm(w)

    ax.plot(x_true, t_true, 'g-', lw=2, label='True sin(2πx)')
    ax.scatter(x_train, t_train, s=80, facecolors='none', edgecolors='blue', lw=2)
    ax.plot(x_true, y_fit, 'r-', lw=2.5, label=label)
    ax.set_title(f'{label}\n$E_{{RMS}}={E_rms:.3f}$, $\\|\\mathbf{{w}}\\|={w_norm:.2f}$', fontsize=11)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.8, 1.8)
    ax.legend(fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('t')

plt.tight_layout()
plt.show()

print('💡 Key Insight:')
print('  λ → 0 : No regularization   → High risk of overfitting')
print('  λ → ∞ : Over-regularization → Underfitting (model too simple)')
print('  Optimal λ : Determined via the validation set!')

---
## 📌 SECTION 2: Probability Theory
### 2.1 Joint, Marginal, and Conditional Probability

- **Sum Rule:** $p(X) = \sum_Y p(X, Y)$
- **Product Rule:** $p(X, Y) = p(Y|X)\, p(X)$
- **Bayes' Theorem:** $p(Y|X) = \dfrac{p(X|Y)\,p(Y)}{p(X)}$

In [ ]:
# Joint distribution simulation (as shown in the lecture slides)
np.random.seed(0)
N_samples = 60

# Class Y=1 and Y=2
x_y1 = np.random.normal(2, 1.2, 35)   # Y=1: low X values
x_y2 = np.random.normal(5, 1.2, 25)   # Y=2: high X values
y1_vals = np.ones(35)
y2_vals = 2 * np.ones(25)

x_all = np.concatenate([x_y1, x_y2])
y_all = np.concatenate([y1_vals, y2_vals])

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Joint, Marginal, and Conditional Probability Visualisation', fontsize=14, fontweight='bold')

# Top-left: p(X, Y) — joint distribution
ax = axes[0, 0]
ax.scatter(x_y1, y1_vals, color='blue', s=50, alpha=0.7, label='Y=1')
ax.scatter(x_y2, y2_vals, color='blue', s=50, alpha=0.7, label='Y=2')
ax.set_yticks([1, 2])
ax.set_yticklabels(['Y=1', 'Y=2'])
ax.set_title('$p(X, Y)$ — Joint Distribution', fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Y')

# Top-right: p(Y) — marginal distribution
ax = axes[0, 1]
p_y1 = len(x_y1) / N_samples
p_y2 = len(x_y2) / N_samples
ax.barh([1, 2], [p_y1, p_y2], color=['#7986CB', '#5C6BC0'], height=0.5)
ax.set_title('$p(Y)$ — Marginal Probability', fontweight='bold')
ax.set_xlabel('Probability'); ax.set_yticks([1, 2])
ax.set_yticklabels(['Y=1', 'Y=2'])
ax.set_xlim(0, 1)
for v, p in zip([1, 2], [p_y1, p_y2]):
    ax.text(p + 0.02, v, f'{p:.2f}', va='center', fontsize=12)

# Bottom-left: p(X) — marginal distribution (sum rule)
ax = axes[1, 0]
ax.hist(x_all, bins=15, color='#7986CB', edgecolor='white', density=True, alpha=0.8)
x_line = np.linspace(x_all.min() - 1, x_all.max() + 1, 200)
p_x_marginal = (p_y1 * norm.pdf(x_line, 2, 1.2) +
                p_y2 * norm.pdf(x_line, 5, 1.2))
ax.plot(x_line, p_x_marginal, 'r-', lw=2, label=r'$p(X) = \sum_Y p(X,Y)$')
ax.set_title('$p(X)$ — Marginal Distribution (Sum Rule)', fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Density')
ax.legend()

# Bottom-right: p(X|Y=1) — conditional distribution
ax = axes[1, 1]
ax.hist(x_y1, bins=12, color='#EF9A9A', edgecolor='white', density=True,
        alpha=0.8, label='Y=1 observations')
x_line_y1 = np.linspace(x_y1.min() - 1, x_y1.max() + 1, 200)
ax.plot(x_line_y1, norm.pdf(x_line_y1, 2, 1.2), 'r-', lw=2, label='Fitted $p(X|Y=1)$')
ax.set_title('$p(X|Y=1)$ — Conditional Distribution', fontweight='bold')
ax.set_xlabel('X'); ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.show()

print('\n📐 Sum Rule Verification:')
print(f'  p(Y=1) = {p_y1:.4f},  p(Y=2) = {p_y2:.4f}')
print(f'  p(Y=1) + p(Y=2) = {p_y1 + p_y2:.4f} ≈ 1.0 ✅')

---
### 2.2 Bayes' Theorem — Medical Diagnosis Example

$$p(Y|X) = \frac{p(X|Y)\,p(Y)}{p(X)}$$

**Real-World Example:** A disease test
- $C_1$: Diseased, $C_2$: Healthy  
- The test came back positive — what is the probability of actually being diseased?

In [ ]:
# Bayes' Theorem — Medical Diagnosis
def bayes_diagnosis(p_disease, sensitivity, specificity):
    """
    Parameters
    ----------
    p_disease   : P(C1) - disease prevalence (prior)
    sensitivity : P(test+ | C1) - true positive rate
    specificity : P(test- | C2) - true negative rate

    Returns
    -------
    (P(C1|test+), P(C2|test+)) — posterior probabilities
    """
    p_healthy           = 1 - p_disease
    p_pos_given_sick    = sensitivity
    p_pos_given_healthy = 1 - specificity

    # Total probability (normalisation): p(X=+)
    p_positive = (p_pos_given_sick    * p_disease +
                  p_pos_given_healthy * p_healthy)

    # Bayes' theorem: P(C1 | X=+)
    p_sick_given_pos    = (p_pos_given_sick    * p_disease) / p_positive
    p_healthy_given_pos = (p_pos_given_healthy * p_healthy) / p_positive

    return p_sick_given_pos, p_healthy_given_pos

# Parameters
p_disease   = 0.01   # Disease prevalence: 1%
sensitivity = 0.95   # Sensitivity: 95%
specificity = 0.90   # Specificity: 90%

p_sick_pos, p_healthy_pos = bayes_diagnosis(p_disease, sensitivity, specificity)

print('🏥 MEDICAL DIAGNOSIS — BAYES THEOREM APPLICATION')
print('=' * 50)
print(f'\n📊 Prior Information:')
print(f'   P(Diseased) = {p_disease:.2f}  ({p_disease*100:.0f}%)')
print(f'   P(Healthy)  = {1-p_disease:.2f}  ({(1-p_disease)*100:.0f}%)')
print(f'\n🔬 Test Performance:')
print(f'   Sensitivity = {sensitivity:.2f}')
print(f'   Specificity = {specificity:.2f}')
print(f'\n✅ TEST CAME BACK POSITIVE — Posterior Probabilities:')
print(f'   P(Diseased | test+) = {p_sick_pos:.4f}  ({p_sick_pos*100:.1f}%)')
print(f'   P(Healthy  | test+) = {p_healthy_pos:.4f}  ({p_healthy_pos*100:.1f}%)')
print(f'\n💡 Surprise! Despite a positive test, the probability of')
print(f'   actually being diseased is only {p_sick_pos*100:.1f}%!')
print(f'   This is due to the low disease prevalence (base rate fallacy).')

# Show how the posterior changes across different prevalence values
prevalences = np.linspace(0.001, 0.5, 100)
posteriors  = [bayes_diagnosis(p, sensitivity, specificity)[0] for p in prevalences]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prevalences * 100, [p * 100 for p in posteriors], 'b-', lw=2.5)
ax.axvline(x=p_disease * 100, color='red', ls='--', alpha=0.7,
           label=f'Current prevalence = {p_disease*100:.0f}%')
ax.axhline(y=p_sick_pos * 100, color='green', ls='--', alpha=0.7,
           label=f'Posterior = {p_sick_pos*100:.1f}%')
ax.scatter([p_disease * 100], [p_sick_pos * 100], color='red', s=100, zorder=5)
ax.set_xlabel('Disease Prevalence P(C₁) [%]', fontsize=12)
ax.set_ylabel('P(Diseased | Test+) [%]', fontsize=12)
ax.set_title('Bayes\' Theorem: Prevalence vs. Posterior Probability\n'
             '(Sensitivity=95%, Specificity=90%)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
### 2.3 Gaussian (Normal) Distribution

$$\mathcal{N}(x|\mu, \sigma^2) = \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{-\frac{1}{2\sigma^2}(x-\mu)^2\right\}$$

**Expected value:** $\mathbb{E}[x] = \mu$  
**Variance:** $\text{var}[x] = \sigma^2$

In [ ]:
# Gaussian Distribution Visualisation
x_range = np.linspace(-5, 8, 500)

params = [
    {'mu': 0, 'sigma2': 1,   'color': '#2196F3', 'label': r'$\mu=0,\ \sigma^2=1$'},
    {'mu': 2, 'sigma2': 0.5, 'color': '#E91E63', 'label': r'$\mu=2,\ \sigma^2=0.5$'},
    {'mu': 0, 'sigma2': 3,   'color': '#4CAF50', 'label': r'$\mu=0,\ \sigma^2=3$'},
    {'mu': 3, 'sigma2': 2,   'color': '#FF9800', 'label': r'$\mu=3,\ \sigma^2=2$'},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gaussian (Normal) Distribution', fontsize=14, fontweight='bold')

# Left: PDF
for p in params:
    y = norm.pdf(x_range, p['mu'], np.sqrt(p['sigma2']))
    axes[0].plot(x_range, y, color=p['color'], lw=2.5, label=p['label'])
    sigma = np.sqrt(p['sigma2'])
    axes[0].axvspan(p['mu'] - sigma, p['mu'] + sigma, alpha=0.05, color=p['color'])

axes[0].set_title('Probability Density Function $p(x)$', fontsize=12)
axes[0].set_xlabel('x'); axes[0].set_ylabel('p(x)')
axes[0].legend(fontsize=10)

# Right: CDF
# FIX: raw string (r"") ensures LaTeX \int is rendered correctly
for p in params:
    y_cdf = norm.cdf(x_range, p['mu'], np.sqrt(p['sigma2']))
    axes[1].plot(x_range, y_cdf, color=p['color'], lw=2.5, label=p['label'])

axes[1].set_title(r'Cumulative Distribution Function $P(z)=\int_{-\infty}^{z}p(x)\,dx$', fontsize=12)
axes[1].set_xlabel('z'); axes[1].set_ylabel('P(z)')
axes[1].legend(fontsize=10)
axes[1].axhline(y=0.5, color='gray', ls=':', alpha=0.5)

plt.tight_layout()
plt.show()

# Numerical Verification
print('🔢 Numerical Verification: E[x] and Var[x]')
print('-' * 55)
rng_verify = np.random.default_rng(0)
for p in params:
    samples = rng_verify.normal(p['mu'], np.sqrt(p['sigma2']), 100_000)
    print(f"  mu={p['mu']}, sigma2={p['sigma2']}  "
          f"→  E[x]={samples.mean():.3f} (expected {p['mu']}), "
          f"Var[x]={samples.var():.3f} (expected {p['sigma2']})")

---
### 2.4 Maximum Likelihood Estimation (MLE)

Log-likelihood function:
$$\ln p(\mathbf{x}|\mu, \sigma^2) = -\frac{1}{2\sigma^2} \sum_{n=1}^{N}(x_n - \mu)^2 - \frac{N}{2}\ln\sigma^2 - \frac{N}{2}\ln(2\pi)$$

MLE solutions:
$$\mu_{ML} = \frac{1}{N}\sum_{n=1}^{N} x_n \qquad \sigma^2_{ML} = \frac{1}{N}\sum_{n=1}^{N}(x_n - \mu_{ML})^2$$

> ⚠️ $\sigma^2_{ML}$ is a **biased** estimator: it systematically underestimates the true variance.

In [ ]:
# MLE Visualisation
np.random.seed(7)
mu_true    = 3.0
sigma_true = 1.5
N_mle = 50
x_obs = np.random.normal(mu_true, sigma_true, N_mle)

# MLE estimates
mu_mle          = np.mean(x_obs)
sigma2_mle      = np.var(x_obs)         # Biased:   1/N
sigma2_unbiased = np.var(x_obs, ddof=1) # Unbiased: 1/(N-1)

print('📊 MLE ESTIMATES')
print('=' * 45)
print(f'True values     : μ = {mu_true:.2f},  σ² = {sigma_true**2:.2f}')
print(f'MLE estimates   : μ_ML = {mu_mle:.4f},  σ²_ML = {sigma2_mle:.4f}')
print(f'Unbiased estim. : σ²_unbiased = {sigma2_unbiased:.4f}  (N/(N-1) × σ²_ML)')
# FIX: Cyrillic character in comment replaced with ASCII
print(f'\n⚠️  MLE variance estimate is BIASED!')
print(f'   σ²_ML underestimates the true variance by {sigma_true**2 - sigma2_mle:.4f}.')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('MLE: Gaussian Parameter Estimation', fontsize=14, fontweight='bold')

# Left: Observations and fitted distribution
ax = axes[0]
x_plot = np.linspace(x_obs.min() - 1, x_obs.max() + 1, 300)
ax.hist(x_obs, bins=12, density=True, alpha=0.5, color='skyblue',
        edgecolor='white', label='Observations')
ax.plot(x_plot, norm.pdf(x_plot, mu_true, sigma_true), 'g-', lw=2.5,
        label=f'True: μ={mu_true}, σ={sigma_true}')
ax.plot(x_plot, norm.pdf(x_plot, mu_mle, np.sqrt(sigma2_mle)), 'r--', lw=2.5,
        label=f'MLE: μ={mu_mle:.2f}, σ={np.sqrt(sigma2_mle):.2f}')
ax.scatter(x_obs, np.zeros_like(x_obs), color='blue', s=20, alpha=0.5,
           zorder=5, label='Data points')
ax.set_xlabel('x'); ax.set_ylabel('Density')
ax.set_title('Fitted Gaussian Distribution')
ax.legend(fontsize=9)

# Right: MLE convergence behaviour as N grows
ax = axes[1]
N_vals = np.arange(5, 501, 5)
mu_estimates     = []
sigma2_estimates = []
np.random.seed(42)
big_sample = np.random.normal(mu_true, sigma_true, 500)

for n in N_vals:
    sub = big_sample[:n]
    mu_estimates.append(np.mean(sub))
    sigma2_estimates.append(np.var(sub))

ax.plot(N_vals, mu_estimates,     'b-', lw=1.5, label='μ_ML estimate')
ax.axhline(y=mu_true,            color='blue', ls='--', alpha=0.6,
           label=f'True μ={mu_true}')
ax.plot(N_vals, sigma2_estimates, 'r-', lw=1.5, label='σ²_ML estimate')
ax.axhline(y=sigma_true**2,      color='red', ls='--', alpha=0.6,
           label=f'True σ²={sigma_true**2}')
ax.set_xlabel('Number of Samples N')
ax.set_ylabel('Parameter Estimate')
ax.set_title('MLE Convergence as N → ∞')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
### 2.5 Likelihood, Prior, and Posterior

$$\underbrace{p(\mathbf{w}|\mathcal{D})}_{\text{Sonsal}} = \frac{\underbrace{p(\mathcal{D}|\mathbf{w})}_{\text{Olabilirlik}} \cdot \underbrace{p(\mathbf{w})}_{\text{Önsel}}}{\underbrace{p(\mathcal{D})}_{\text{Normalizasyon}}}$$

**In words:** posterior ∝ likelihood × prior

In [ ]:
# Bayesian Updating — Coin Flip Example
# θ = probability of heads; observations: 1=heads, 0=tails

theta = np.linspace(0, 1, 500)

# Prior: uniform — no prior knowledge
prior = np.ones_like(theta)
prior /= prior.sum()

observations = [1, 1, 0, 1, 1, 0, 1, 1, 1, 0]  # 7 heads, 3 tails

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Bayesian Updating: Coin Flip\n(Prior → Posterior)',
             fontsize=14, fontweight='bold')

posterior = prior.copy()
heads, tails = 0, 0

for idx, (ax, obs) in enumerate(zip(axes.flat, observations)):
    likelihood = (theta ** obs) * ((1 - theta) ** (1 - obs))
    posterior  = posterior * likelihood
    posterior /= posterior.sum()  # normalise

    if obs == 1:
        heads += 1
    else:
        tails += 1

    map_estimate = theta[np.argmax(posterior)]
    mle_estimate = heads / (heads + tails)

    ax.fill_between(theta, posterior, alpha=0.4, color='#5C6BC0')
    ax.plot(theta, posterior, '#3949AB', lw=2)
    ax.axvline(x=map_estimate, color='red',   ls='--', lw=1.5, label=f'MAP={map_estimate:.2f}')
    ax.axvline(x=mle_estimate, color='green', ls=':',  lw=1.5, label=f'MLE={mle_estimate:.2f}')
    ax.axvline(x=0.6,          color='gray',  ls='-',  lw=1,   alpha=0.5, label='True=0.6')

    result = '✅ Heads' if obs == 1 else '❌ Tails'
    ax.set_title(f'Flip {idx+1}: {result}\nH={heads}, T={tails}', fontsize=9)
    ax.set_xlabel(r'$\theta$', fontsize=9); ax.set_ylabel(r'$p(\theta|D)$', fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print('\n💡 As the Number of Observations Grows:')
print('   → The influence of the prior diminishes')
print('   → The posterior narrows (uncertainty decreases)')
print('   → The MAP estimate converges toward the MLE estimate')

---
## 📌 SECTION 3: Decision Theory
### 3.1 Minimum-Error Classification — Decision Boundaries

To achieve minimum error probability, assign each x to the class with the **highest posterior**:

$$\hat{C}(x) = \arg\max_k\ p(C_k|x)$$

In [ ]:
# Decision Boundary Visualisation — 1D
x_plot = np.linspace(-2, 10, 500)

# Two-class parameters
mu1, sigma1, p_c1 = 2.5, 1.2, 0.45   # C1: Diseased
mu2, sigma2, p_c2 = 6.0, 1.5, 0.55   # C2: Healthy

# Class-conditional densities
p_x_c1 = norm.pdf(x_plot, mu1, sigma1)
p_x_c2 = norm.pdf(x_plot, mu2, sigma2)

# Joint distributions: p(x, Ck) = p(x|Ck) * p(Ck)
joint_c1 = p_x_c1 * p_c1
joint_c2 = p_x_c2 * p_c2

# Posterior probabilities
p_x        = joint_c1 + joint_c2
posterior_c1 = joint_c1 / p_x
posterior_c2 = joint_c2 / p_x

# Decision boundary: p(C1|x) = p(C2|x)
boundary_idx = np.argmin(np.abs(posterior_c1 - posterior_c2))
x0 = x_plot[boundary_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Decision Theory: Minimum-Error Decision Boundary', fontsize=14, fontweight='bold')

# Left: Joint distributions
ax = axes[0]
ax.plot(x_plot, joint_c1, 'b-', lw=2.5, label='$p(x, C_1)$ - Diseased')
ax.plot(x_plot, joint_c2, 'r-', lw=2.5, label='$p(x, C_2)$ - Healthy')
ax.axvline(x=x0, color='black', ls='--', lw=2, label=f'Decision boundary $x_0={x0:.2f}$')

idx_left  = x_plot < x0
idx_right = x_plot > x0
ax.fill_between(x_plot[idx_left],  joint_c2[idx_left],  alpha=0.35, color='red',  label='Hata: $C_2 \\to R_1$')
ax.fill_between(x_plot[idx_right], joint_c1[idx_right], alpha=0.35, color='blue', label='Hata: $C_1 \\to R_2$')

ax.set_xlabel('x', fontsize=12); ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Joint Distributions $p(x, C_k)$')
ax.legend(fontsize=9)
# FIX: Unicode arrow '→' replaced with LaTeX \rightarrow
ax.text(0.05, -0.12, r'$\mathcal{R}_1\ (\rightarrow C_1)$', transform=ax.transAxes,
        ha='left', fontsize=12)
ax.text(0.65, -0.12, r'$\mathcal{R}_2\ (\rightarrow C_2)$', transform=ax.transAxes,
        ha='left', fontsize=12)

# Right: Posterior probabilities
ax = axes[1]
ax.plot(x_plot, posterior_c1, 'b-', lw=2.5, label='$p(C_1|x)$ - Diseased')
ax.plot(x_plot, posterior_c2, 'r-', lw=2.5, label='$p(C_2|x)$ - Healthy')
ax.axvline(x=x0, color='black', ls='--', lw=2, label=f'$x_0 = {x0:.2f}$')
ax.axhline(y=0.5, color='gray', ls=':', alpha=0.5)
ax.fill_between(x_plot, 0, 1, where=(posterior_c1 > posterior_c2),
                alpha=0.1, color='blue', label=r'$\hat{C}=C_1$')
ax.fill_between(x_plot, 0, 1, where=(posterior_c2 > posterior_c1),
                alpha=0.1, color='red',  label=r'$\hat{C}=C_2$')
ax.set_xlabel('x', fontsize=12); ax.set_ylabel('Posterior Probability', fontsize=12)
ax.set_title('Posterior Probabilities $p(C_k|x)$')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

# FIX: np.trapz → np.trapezoid (removed in NumPy 2.0)
error_c2_in_r1 = np.trapezoid(joint_c2[idx_left],  x_plot[idx_left])
error_c1_in_r2 = np.trapezoid(joint_c1[idx_right], x_plot[idx_right])
total_error    = error_c2_in_r1 + error_c1_in_r2

print(f'\n📐 Decision Boundary: x₀ = {x0:.3f}')
print(f'📊 Error Analysis:')
print(f'   p(x ∈ R₁, C₂) = {error_c2_in_r1:.4f}  (C₂ → R₁ error)')
print(f'   p(x ∈ R₂, C₁) = {error_c1_in_r2:.4f}  (C₁ → R₂ error)')
print(f'   Total p(error) = {total_error:.4f}')

---
### 3.2 Reject Option

Withhold a decision in regions where posteriors are ambiguous, thereby reducing the error rate.

In [ ]:
# Reject Option Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Reject Option', fontsize=14, fontweight='bold')

thresholds    = [0.6, 0.8]
colors_reject = ['#FF9800', '#F44336']

for ax, theta_t, col in zip(axes, thresholds, colors_reject):
    ax.plot(x_plot, posterior_c1, 'b-', lw=2.5, label='$p(C_1|x)$')
    ax.plot(x_plot, posterior_c2, 'r-', lw=2.5, label='$p(C_2|x)$')
    ax.axhline(y=theta_t, color=col, ls='--', lw=2, label=f'Threshold θ={theta_t}')

    max_posterior = np.maximum(posterior_c1, posterior_c2)
    reject_mask   = max_posterior < theta_t
    accept_mask   = ~reject_mask

    ax.fill_between(x_plot, 0, 1, where=reject_mask,
                    alpha=0.25, color=col, label='Rejection Region')
    ax.fill_between(x_plot, 0, 1, where=(accept_mask & (posterior_c1 > posterior_c2)),
                    alpha=0.1, color='blue', label=r'$\rightarrow C_1$')
    ax.fill_between(x_plot, 0, 1, where=(accept_mask & (posterior_c2 > posterior_c1)),
                    alpha=0.1, color='red',  label=r'$\rightarrow C_2$')

    rejected_pct = reject_mask.mean() * 100
    ax.set_title(f'θ = {theta_t}  |  Rejected: ~{rejected_pct:.1f}%', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('Sonsal Olasılık')
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('💡 Trade-off:')
print('   Higher θ → more samples rejected, lower error on accepted samples')
print('   Lower θ  → fewer samples rejected, higher error risk')

---
### 3.3 Loss Matrix

$$\mathbb{E}[L] = \sum_k \sum_j \int_{\mathcal{R}_j} L_{kj}\, p(\mathbf{x}, C_k)\, d\mathbf{x}$$

The cost of predicting $C_j$ when the true class is $C_k$ is $L_{kj}$.

In [ ]:
# Loss Matrix Visualisation — Medical Diagnosis Scenario
# Classes: C1=Cancer, C2=Healthy  |  Predictions: R1=Cancer, R2=Healthy
# L[true, predicted]
L_symmetric  = np.array([[0, 1],
                          [1, 0]])

L_asymmetric = np.array([[0, 10],   # Missing cancer is 10× more costly!
                          [1,  0]])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Loss Matrix Comparison', fontsize=14, fontweight='bold')

class_labels = ['$C_1$ (Cancer)', '$C_2$ (Healthy)']
pred_labels  = ['$R_1$ (→Cancer)', '$R_2$ (→Healthy)']

for ax, L, title in zip(axes,
                         [L_symmetric, L_asymmetric],
                         ['Symmetric Loss\n(All errors equally costly)',
                          'Asymmetric Loss\n(Missing cancer is 10× more costly)']):
    im = ax.imshow(L, cmap='YlOrRd', vmin=0, vmax=L.max())
    ax.set_xticks([0, 1]); ax.set_xticklabels(pred_labels, fontsize=11)
    ax.set_yticks([0, 1]); ax.set_yticklabels(class_labels, fontsize=11)
    ax.set_xlabel('Predicted Class', fontsize=11)
    ax.set_ylabel('True Class', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')

    for i in range(2):
        for j in range(2):
            color = 'white' if L[i, j] > L.max() / 2 else 'black'
            ax.text(j, i, f'{L[i, j]}', ha='center', va='center',
                    fontsize=20, fontweight='bold', color=color)
    plt.colorbar(im, ax=ax, label='Loss $L_{kj}$')

plt.tight_layout()
plt.show()

print('\n🔍 Effect of Asymmetric Loss:')
print('   Symmetric : Misclassifying either class carries equal cost')
print('   Asymmetric: Classifying cancer as "healthy" is 10× more costly')
print('   → With asymmetric loss, the boundary shifts toward C2 (more cautious diagnosis)')

---
### 3.4 Generative, Discriminative, and Discriminant Models

| Model Type | Approach | Example |
|------------|----------|-------|
| **Generative** | $p(x\|C_k)$ and $p(C_k)$ → $p(C_k\|x)$ | Naive Bayes, GMM |
| **Discriminative** | Directly models $p(C_k\|x)$ | Logistic Regression |
| **Discriminant** | $f(x)$ function | SVM, Perceptron |

In [ ]:
# 2D Classification: Comparison of Three Approaches
np.random.seed(0)
X_cls, y_cls = make_classification(
    n_samples=200, n_features=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

models = [
    ('Generative Model\n(Gaussian Naive Bayes)',   GaussianNB()),
    ('Discriminative Model\n(Logistic Regression)', LogisticRegression(random_state=0)),
    ('Discriminant Function\n(Linear SVM)',      SVC(kernel='linear', probability=True, random_state=0)),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Generative / Discriminative / Discriminant Model Comparison',
             fontsize=14, fontweight='bold')

colors_2d = ['#2196F3', '#E91E63']
x_min, x_max = X_cls[:, 0].min() - 0.5, X_cls[:, 0].max() + 0.5
y_min, y_max = X_cls[:, 1].min() - 0.5, X_cls[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

for ax, (name, clf) in zip(axes, models):
    clf.fit(X_cls, y_cls)
    acc = clf.score(X_cls, y_cls)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=2, levels=[0.5])

    for cls_id, color in zip([0, 1], colors_2d):
        mask = y_cls == cls_id
        ax.scatter(X_cls[mask, 0], X_cls[mask, 1], c=color, s=40,
                   edgecolors='white', lw=0.5, label=f'$C_{cls_id+1}$')

    ax.set_title(f'{name}\nAccuracy: {acc*100:.1f}%', fontsize=11)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print('\n📊 Summary of the Three Approaches:')
print('─' * 60)
print(f'{"Model":<30} {"Approach":<20} {"Probabilistic?"}')
print('─' * 60)
print(f'{"Gaussian Naive Bayes":<30} {"Generative":<20} Yes (full joint)')
print(f'{"Logistic Regression":<30} {"Discriminative":<20} Yes (conditional)')
print(f'{"SVM (Linear)":<30} {"Discriminant":<20} No (score-based)')

---
## 📌 SECTION 4: Summary and Types of Learning

In [ ]:
# Machine Learning Types — Summary Diagram
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Machine Learning Approaches — Comprehensive Summary',
             fontsize=16, fontweight='bold', pad=20)

boxes = [
    (0.3, 3.8, 2.8, 2.8, 'Supervised Learning',
     'Labelled data\n(x, t) pairs\n\n• Classification\n• Regression\n\nExample: Digit recognition',
     '#BBDEFB'),
    (3.6, 3.8, 2.8, 2.8, 'Unsupervised Learning',
     'Unlabelled data\nOnly x vectors\n\n• Clustering\n• Density Estimation\n• Dimensionality Reduction',
     '#C8E6C9'),
    (6.9, 3.8, 2.8, 2.8, 'Reinforcement Learning',
     'Reward maximisation\n\n• Exploration\n• Exploitation\n\nExample: Game playing',
     '#FFE0B2'),
    (0.3, 0.3, 2.8, 3.0, 'Probability Theory',
     '• Sum Rule\n• Product Rule\n• Bayes Theorem\n• MLE\n• Gaussian Distribution',
     '#E1BEE7'),
    (3.6, 0.3, 2.8, 3.0, 'Decision Theory',
     '• Decision Boundaries\n• Loss Matrix\n• Min. Error = Max. Posterior\n• Reject Option',
     '#FFCDD2'),
    (6.9, 0.3, 2.8, 3.0, 'Model Selection',
     '• Curve Fitting\n• Overfitting/Underfitting\n• Regularization (λ)\n• Validation Set',
     '#B2EBF2'),
]

for (x, y, w, h, title, content, color) in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.3, title,   ha='center', va='top',
            fontsize=11, fontweight='bold', color='#1A237E')
    ax.text(x + w/2, y + h - 0.7, content, ha='center', va='top',
            fontsize=9, color='#212121', linespacing=1.5)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Lecture Summary ───
print('=' * 60)
print('  📚 LECTURE 1 SUMMARY — Key Formulae')
print('=' * 60)

formulas = [
    ('Polynomial Regression',
     'y(x,w) = Σ wⱼ xʲ  (j=0..M)'),
    ('Error Function (Least Squares)',
     'E(w) = ½ Σ {y(xₙ,w) - tₙ}²'),
    ('RMS Error',
     'E_RMS = √(mean((y-t)²))'),
    ('Regularization (Ridge)',
     'E~(w) = E(w) + (λ/2)||w||²'),
    ('Sum Rule',
     'p(X) = Σ_Y p(X, Y)'),
    ('Product Rule',
     'p(X, Y) = p(Y|X) p(X)'),
    ('Bayes Theorem',
     'p(Y|X) = p(X|Y) p(Y) / p(X)'),
    ('Posterior ∝',
     'p(w|D) ∝ p(D|w) · p(w)'),
    ('Gaussian Distribution',
     'N(x|μ,σ²) = (2πσ²)^(-½) exp{-(x-μ)²/(2σ²)}'),
    ('MLE — Mean',
     'μ_ML = (1/N) Σ xₙ'),
    ('MLE — Variance (biased)',
     'σ²_ML = (1/N) Σ (xₙ - μ_ML)²'),
    ('Unbiased Variance',
     'σ²_unbiased = (1/(N-1)) Σ (xₙ - μ_ML)²'),
    ('Min. Error Rule',
     'C_hat(x) = argmax_k p(Cₖ|x)'),
    ('Expected Loss',
     'E[L] = Σ_k Σ_j ∫_Rⱼ Lₖⱼ p(x,Cₖ) dx'),
]

for i, (name, formula) in enumerate(formulas, 1):
    print(f'\n  {i:>2}. {name}')
    print(f'      {formula}')

print('\n' + '=' * 60)
print('  ✅ Notebook complete!')
print('=' * 60)

---
## 🎯 Exercises

1. **Curve Fitting:** Fit polynomials of degree M=0,3,6,9 to N=20 data points. Compare with the N=10 case. Does the overfitting behaviour change?

2. **Regularization:** Sweep λ from 10⁻⁵ to 10⁵ and compute both training and test RMS errors. What is the "ideal" λ?

3. **Bayesian Updating:** Use an informative prior (e.g. Beta(5,5)) in the coin-flip example. Compare with the uniform prior.

4. **Loss Matrix:** How does the decision boundary shift when L₁₂=5 and L₂₁=1 (false negatives are very costly)?

5. **Model Comparison:** Compare the accuracy of the Generative, Discriminative, and Discriminant models across different `class_sep` values.

---
*Generative Artificial Intelligence — Haydar Kılıç, Artificial Intelligence Engineering*